# AIoT sensors: from InfluxDB to a forecast

In this notebook you:

1. List the sensors that have sent data
2. Load the readings of one sensor into a pandas table, and plot them
3. Flag unusual readings with machine learning (scikit-learn)
4. Forecast the next readings (sktime)
5. Ask the gateway's MCP server for data, from Python (an MCP client)

**How to run it:** click a code cell and press **Shift+Enter**. That runs the
cell and moves to the next one. Run the cells from top to bottom: later cells
use variables made by earlier ones. To start again: Kernel, Restart Kernel
and Run All Cells.

`aiot_data.py`, next to this notebook, does the database queries, so you only
call `aiot_data.sensors()` and `aiot_data.readings(...)`.

In [ ]:
# Libraries
import aiot_data                    # our helper for the sensor database (aiot_data.py)
import matplotlib.pyplot as plt     # charts
import pandas as pd                 # tables (DataFrames)

# Where the data comes from
print("InfluxDB:", aiot_data.INFLUX_URL, "database", aiot_data.INFLUX_DB)
print("Times are shown in:", aiot_data.TIMEZONE)

# A table with one row per device that sent data in the last 24 hours.
# Columns: sensor_uid (sensor ID), mac (Device ID), fields (what it measures),
# last (its latest values).
devices = aiot_data.sensors("24h")

# The last line of a cell is shown as the cell's output: here, the table
devices

## 1. Pick a sensor and field

The next cell picks a sensor automatically: the first one that measures
temperature (`temperature_c`), otherwise the first sensor in the table.

To choose yourself, replace the cell with for example:

```python
UID = "1:1"                 # the sensor ID, from the table above
FIELD = "temperature_c"     # one of its fields
MAC = None                  # None = every device of the sensor, or one Device ID
WINDOW = "6h"               # how far back: "15m", "1h", "6h", "24h", "7d"
df = aiot_data.readings(UID, FIELD, MAC, WINDOW)
```

In [ ]:
# Start with the first device in the table (row 0) ...
chosen = devices.iloc[0]

# ... but if a device measures temperature, take the first one of those.
# iterrows() goes through the table one row at a time.
for index, device in devices.iterrows():
    if "temperature_c" in device["fields"]:
        chosen = device
        break                       # found one: stop looking

UID = chosen["sensor_uid"]          # e.g. "1:1"
MAC = chosen["mac"]                 # its Device ID

# Use temperature if this device has it, otherwise its first field
if "temperature_c" in chosen["fields"]:
    FIELD = "temperature_c"
else:
    FIELD = chosen["fields"][0]

WINDOW = "6h"                       # the last 6 hours

# Read the readings: a table with the columns time, mac, value, status.
# The times are in local time (aiot_data.TIMEZONE, Asia/Kolkata = IST).
df = aiot_data.readings(UID, FIELD, MAC, WINDOW)
print(f"{UID} {MAC} {FIELD}: {len(df)} readings in the last {WINDOW}")

# .tail() shows the last 5 rows (the newest readings)
df.tail()

Now plot the readings: time on the horizontal axis, the value on the
vertical axis.

In [ ]:
# df.plot draws a line chart from the table; it returns the chart's
# axes ("ax"), which we use to add labels.
ax = df.plot(x="time", y="value", figsize=(10, 3), legend=False)
ax.set_ylabel(FIELD)                # the label on the vertical axis
ax.set_title(f"{UID} {MAC}")        # the title above the chart
plt.show()                          # draw it

## 2. Unusual readings (scikit-learn)

A sensor sometimes gives a strange value: a spike, a sudden jump, a
glitch. Here a machine-learning model finds them for us.

The model is an **Isolation Forest**. It is *unsupervised*: we do not tell
it which readings are wrong. It looks at all readings and finds the ones
that are easiest to separate from the rest, i.e. the rarest.

It gets two numbers per reading:

- **value**: the reading itself
- **step**: how much it changed since the previous reading (a sudden jump
  is suspicious even when the value itself looks normal)

`contamination=0.02` tells it to flag the rarest 2 % of the readings.

In [ ]:
from sklearn.ensemble import IsolationForest

# step = the change since the previous reading.
# diff() subtracts each value from the one before it. The first reading has
# no previous one, so diff() gives NaN ("not a number"); fillna(0) makes it 0.
step = df["value"].diff()
step = step.fillna(0)

# X = the input for the model: a table with one row per reading and the two
# numbers as columns. (In machine learning the input is usually called X.)
X = pd.DataFrame()
X["value"] = df["value"]
X["step"] = step

# Create the model and let it learn from the readings
model = IsolationForest(contamination=0.02, random_state=0)   # random_state: same result every run
model.fit(X)

# predict() answers per reading: 1 = normal, -1 = unusual
prediction = model.predict(X)

# Add a True/False column "unusual" to the readings table
df["unusual"] = prediction == -1

# The unusual rows only
unusual = df[df["unusual"]]
print(len(unusual), "unusual readings of", len(df))

# Plot all readings as a line, and the unusual ones as red dots on top
ax = df.plot(x="time", y="value", figsize=(10, 3), legend=False)
unusual.plot.scatter(x="time", y="value", color="red", ax=ax)
ax.set_ylabel(FIELD)
plt.show()

## 3. Forecast (sktime)

Can we predict the next readings? A fair test: hide the last part of the
data, let the forecasters predict it from the earlier part, and compare
their prediction with what really happened.

1. Average the readings per minute (one value per minute is easier to
   forecast than an irregular stream).
2. Split: the first 80 % to **train** (learn from), the last 20 % to
   **test** (check the forecast).
3. Two simple forecasters:
   - **trend**: draws a straight line through the training data and extends it
   - **last 10 min**: predicts the average of the last 10 minutes, every minute
4. **Mean absolute error** (MAE): the average distance between prediction and
   reality, in the field's unit. Smaller is better.

In [ ]:
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.trend import PolynomialTrendForecaster
from sktime.performance_metrics.forecasting import mean_absolute_error

# 1. One value per minute
y = df.set_index("time")            # use the time as the row label (the "index")
y = y["value"]                      # keep only the values
y = y.resample("1min").mean()       # group into 1-minute slots, average each slot
y = y.interpolate()                 # a minute without readings: fill it in between its neighbours

# sktime wants the time as "minute periods" without a time zone
y.index = y.index.tz_localize(None)     # remove the time zone label; the clock times stay IST
y.index = y.index.to_period("min")      # 10:54:00 -> the minute 10:54

if len(y) < 10:
    # too little data for a meaningful test
    print(f"Only {len(y)} minutes of data so far: the forecast needs at least 10.")
    print("Wait a few minutes and run this cell again.")
else:
    # 2. Split in time order: first 80 % train, last 20 % test.
    # (Not randomly: the forecasters may only use the past.)
    y_train, y_test = temporal_train_test_split(y, test_size=0.2)

    # The forecasting horizon: the minutes we want predictions for,
    # i.e. exactly the minutes of the test part
    horizon = ForecastingHorizon(y_test.index, is_relative=False)

    # The "last N minutes" forecaster needs N minutes of training data;
    # use 10, or fewer if the training part is shorter
    window = 10
    if len(y_train) < window:
        window = len(y_train)

    # 3. The two forecasters, in a dictionary: name -> forecaster
    forecasters = {
        "trend": PolynomialTrendForecaster(degree=1),     # degree 1 = straight line
        f"last {window} min": NaiveForecaster(strategy="mean", window_length=window),
    }

    # Plot the real values, then add one line per forecaster
    ax = y.plot(figsize=(10, 3), label="measured")
    for name, forecaster in forecasters.items():
        forecaster.fit(y_train)                     # learn from the training part
        prediction = forecaster.predict(horizon)    # predict the test minutes

        # 4. How far off was it, on average?
        error = mean_absolute_error(y_test, prediction)
        print(f"{name:12s} mean absolute error {error:.3f}")

        prediction.plot(ax=ax, label=name)

    ax.set_ylabel(FIELD)
    ax.legend()                     # the box that names each line
    plt.show()

## 4. MCP client

The gateway runs an **MCP server** (Model Context Protocol). It offers
*tools* such as `list_sensors` and `read_latest`; an AI assistant like
Claude calls these tools to answer questions about the sensors.

Here **Python is the client**: we call the same tools ourselves, without an
AI. In Docker the server is at `http://mcp:8000/mcp` (the `MCP_URL` setting).

**`async` and `await`:** talking to a server takes time. A function marked
`async def` may wait for answers, and `await` is where it waits. In a
notebook you can write `await` directly in a cell, as in the last lines
below.

**`async with`:** opens a connection and closes it again automatically at the
end of the block, even if something goes wrong.

In [ ]:
import json                         # to turn the server's JSON text into Python data

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def list_tools():
    """Ask the MCP server which tools it offers; return their names."""
    # 1. Open the connection to the server (over HTTP)
    async with streamable_http_client(aiot_data.MCP_URL) as (read, write, session_id):
        # 2. Start an MCP session on that connection
        async with ClientSession(read, write) as session:
            await session.initialize()          # say hello to the server
            answer = await session.list_tools() # ask for the list of tools

    # 3. Collect just the names
    names = []
    for tool in answer.tools:
        names.append(tool.name)
    return names


async def call_tool(tool_name, arguments):
    """Call one tool with its arguments; return its answer as Python data."""
    async with streamable_http_client(aiot_data.MCP_URL) as (read, write, session_id):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)

    # The answer is a list of "content" parts; ours has one, holding JSON text
    if len(result.content) == 0:
        return None
    text = result.content[0].text
    data = json.loads(text)             # JSON text -> Python dictionaries and lists
    return data


# Which tools are there?
tools = await list_tools()
print("tools:", tools)

# Call read_latest for our sensor: the newest value of each field, per device
latest = await call_tool("read_latest", {"sensor_uid": UID})

# json.dumps(..., indent=2) prints the data neatly, one item per line
print(json.dumps(latest, indent=2))